# Token-based track: indicators, bag-of-words and character n-grams

**Why:** the rubric asks for the tutorial's two representations (token indicators and bag-of-words). This notebook runs them, plus character n-grams, with logistic regression, an MLP and a residual MLP. It uses the shared protocol, so every result can be compared directly with the neural track in `neural_project.ipynb`.

| | |
|---|---|
| **Features** | Token indicators, bag-of-words, character 2-4-grams |
| **Tokenisers** | LUSR (`roberta-bpe`) and `ours-bpe-8k`; character n-grams use no tokeniser |
| **Models** | `logreg`, `mlp`, `resmlp`, from `shared/eval_harness.py` |
| **Protocol** | 50 shared folds grouped by prompt, nested tuning, balanced accuracy, ROPE = 1 pp |
| **Results** | One JSON per system in `shared/results/` |

## Step 1 · Setup

In [ ]:
import sys
import time

sys.path.insert(0, "..")  # this notebook lives in root/part1, so `shared` and `part1` are importable

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer

from part1.clslib import Encoder
from shared import analysis as A, eval_harness as H
from shared.data import load_data
from shared.tokenization import get_tokenizer, preprocess, tokenize

DATA = load_data()
d = DATA.train
y, groups = DATA.y, DATA.groups
FOLDS = H.load_folds(d["id"].tolist())  # 10 x 5 = 50 folds
print(d.shape, len(DATA.classes), "classes |", len(FOLDS), "folds")

RESULTS = {}  # name -> run_cv result; every model/representation lands here
RECOMPUTE = False  # True: ignore saved results in shared/results/ and re-run every CV


def cv(name, X, model, folds=None, **kw):
    """run_cv + save to shared/results/, or load the saved result (same name, same number of folds)."""
    folds = FOLDS if folds is None else folds
    saved = H.load_results(name.replace("/", "__") + ".json")
    if saved and not RECOMPUTE and len(saved[name]["folds"]) == len(folds):
        RESULTS[name] = saved[name]
        ba = H.fold_scores(RESULTS[name])
        print(f"{name:34s} bal.acc {ba.mean():.4f} ± {ba.std():.4f}  (loaded, {len(ba)} folds)")
    else:
        RESULTS[name] = H.run_cv(X, y, groups, folds, model, name=name, **kw)
        H.save_result(RESULTS[name])
    return RESULTS[name]

(2159, 4) 18 classes | 50 folds


## Step 2 · Features

`TokenIndicator` and `BagOfWords` follow the `Encoder` classes of the tutorial. There are two changes: tokenisers are loaded by name from `shared/tokenization.py`, so all tracks use the same ones, and `__len__` reads `self` (the tutorial's version reads a global `tokenizer`). The tutorial leaves `BagOfWords` as an exercise, so that class is our own.

`encode_all` stores the vectors as a sparse matrix and drops tokens that never occur in the training split. Those columns are always zero, so they only cost memory.

**Fitted on the whole training split:** `ours-bpe-8k`, the dropped tokens and the character n-gram vocabulary all see the validation prompts of each fold, without seeing their labels. This can make their scores slightly optimistic.

In [2]:
def vocab_size(tokenizer_name):
    tokenizer = get_tokenizer(tokenizer_name)
    return tokenizer.get_vocab_size() if hasattr(tokenizer, "get_vocab_size") else len(tokenizer)


class TokenIndicator(Encoder):
    def __init__(self, tokenizer_name):
        self.tokenizer_name = tokenizer_name
        self.size = vocab_size(tokenizer_name)

    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return self.size

    def __call__(self, x: str) -> np.ndarray:
        """Represent x as a D-dimensional vector of real numbers"""
        unique_token_ids = list(set(tokenize([x], self.tokenizer_name)[0]))
        vector = np.zeros(len(self))
        vector[unique_token_ids] = 1
        return vector


class BagOfWords(Encoder):
    def __init__(self, tokenizer_name):
        self.tokenizer_name = tokenizer_name
        self.size = vocab_size(tokenizer_name)

    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return self.size

    def __call__(self, x: str) -> np.ndarray:
        """Represent x as a D-dimensional vector of real numbers"""
        token_ids = tokenize([x], self.tokenizer_name)[0]
        return np.bincount(token_ids, minlength=len(self)).astype(float)


def encode_all(texts, encoder):
    """Stack encoder(text) for every text, dropping tokens that never occur in these texts."""
    X = sp.vstack([sp.csr_matrix(encoder(text)) for text in texts], format="csr")
    return X[:, np.unique(X.indices)]


def char_ngrams(texts, ngram_range=(2, 4), max_features=20_000):
    """Counts of the most frequent character n-grams; case and typography are authorship cues, so both stay."""
    vectorizer = CountVectorizer(analyzer="char", ngram_range=ngram_range, lowercase=False, max_features=max_features)
    return vectorizer.fit_transform([preprocess(text) for text in texts]).astype(float).tocsr()

In [3]:
TOKENIZERS = {"lusr": "roberta-bpe", "ours8k": "ours-bpe-8k"}
texts = d["text"].tolist()

FEATURES = {}
for short_name, tokenizer_name in TOKENIZERS.items():
    FEATURES[f"indicator-{short_name}"] = encode_all(texts, TokenIndicator(tokenizer_name))
    FEATURES[f"bow-{short_name}"] = encode_all(texts, BagOfWords(tokenizer_name))
FEATURES["char-2-4grams"] = char_ngrams(texts)

for name, X in FEATURES.items():
    print(f"{name:20s} {X.shape}  {X.nnz / X.shape[0]:.0f} non-zeros per text")

indicator-lusr       (2159, 16176)  110 non-zeros per text
bow-lusr             (2159, 16176)  110 non-zeros per text
indicator-ours8k     (2159, 7573)  119 non-zeros per text
bow-ours8k           (2159, 7573)  119 non-zeros per text
char-2-4grams        (2159, 20000)  1319 non-zeros per text


## Step 3 · Models

Each model comes from `H.MODELS` with its shared grid. Run the timing cell first: it tries one fold of every system and saves nothing, so you can see what the full run will cost.

In [4]:
# Timing test: one fold of every system, nothing saved
MODELS = ["logreg", "mlp", "resmlp"]

timing = []
for name, X in FEATURES.items():
    for model in MODELS:
        t0 = time.time()
        res = H.run_cv(X, y, groups, FOLDS[:1], model, name=f"{name}/{model}", verbose=False)
        timing.append({"system": f"{name}/{model}", "bal.acc (1 fold)": H.fold_scores(res).mean(), "seconds": time.time() - t0})
        print(timing[-1], flush=True)
display(pd.DataFrame(timing).round(3))

{'system': 'indicator-lusr/logreg', 'bal.acc (1 fold)': np.float64(0.19917471819645732), 'seconds': 12.52722454071045}
{'system': 'indicator-lusr/mlp', 'bal.acc (1 fold)': np.float64(0.05555555555555555), 'seconds': 969.5423686504364}


KeyboardInterrupt: 

In [ ]:
# Full protocol: 50 folds per system. Saved systems are loaded, so you can stop and re-run this cell.
for model in MODELS:
    for name, X in FEATURES.items():
        cv(f"{name}/{model}", X, model, folds=FOLDS)

indicator-lusr/logreg        bal.acc 0.2047 ± 0.0148  (50 folds, 83s)
bow-lusr/logreg                    bal.acc 0.1996 ± 0.0174  (loaded, 50 folds)
indicator-ours8k/logreg      bal.acc 0.2065 ± 0.0150  (50 folds, 47s)
bow-ours8k/logreg                  bal.acc 0.2021 ± 0.0159  (loaded, 50 folds)
char-2-4grams/logreg         bal.acc 0.2602 ± 0.0180  (50 folds, 585s)
indicator-lusr/mlp           bal.acc 0.0569 ± 0.0063  (5 folds, 1612s)
bow-lusr/mlp                 bal.acc 0.0565 ± 0.0098  (5 folds, 2821s)


KeyboardInterrupt: 

## Step 4 · Results

The comparison uses `lusr/logreg`, the neural track's best plain system, as the baseline.

In [ ]:
SYSTEMS = {k: v for k, v in RESULTS.items() if k.split("/")[0] in FEATURES}
display(A.results_table(SYSTEMS))

ALL_RESULTS = {**H.load_results("lusr__logreg.json"), **SYSTEMS}
CMP_TABLE, CMPS = A.comparison_table(ALL_RESULTS, baseline="lusr/logreg")
display(CMP_TABLE)

**Findings:** to be written after the full run.